# BAN 612 — Data Science & AI Job Market Analysis
## Notebook 1: Data Collection & Cleaning

**Team Members:** [Member 1], [Member 2], [Member 3], Miguel Davila

**Objective:** Scrape 500+ job listings across Data Science and AI roles from multiple sources, then clean and organize the data for analysis.

---

### AI Usage Disclosure
As required by the project instructions, we document AI usage throughout this notebook:
- **Claude (Anthropic)** was used to help structure the scraping pipeline, generate boilerplate code for API calls and BeautifulSoup parsing, and assist with data cleaning logic.
- All code was reviewed, tested, and adapted by team members.
---

## 1. Setup & Imports

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import random
import re
import json
from datetime import datetime, timedelta
from urllib.parse import urljoin, quote_plus
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_columns', 20)

# Common headers to avoid 403 errors (per Prof. Wang's notebook)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Define Target Job Titles, Search Terms & Relevance Filter

Per the project instructions, we search two domains:
- **Data Science:** Data Scientist, Data Analyst, Quantitative Analyst, Quantitative Researcher, Business Analyst, Business Intelligence Analyst
- **AI-related:** Machine Learning Engineer, AI Research Scientist, Prompt Engineer, Response Engineer

We apply a **relevance filter at scrape time** to only collect jobs whose titles contain keywords related to these roles. This avoids collecting irrelevant listings (e.g., generic software engineering roles) and keeps the pipeline lean.

In [2]:
# Keywords used to filter job titles at scrape time
RELEVANT_KEYWORDS = [
    'data', 'analyst', 'analytics', 'machine learning', 'ml ',
    'ai ', 'artificial intelligence', 'quantitative', 'quant ',
    'business intelligence', 'prompt engineer', 'deep learning',
    'nlp', 'computer vision', 'bi '
]

def is_relevant(title):
    """Check if a job title matches our target roles."""
    title_lower = title.lower()
    return any(kw in title_lower for kw in RELEVANT_KEYWORDS)

# Search queries to use across sources
SEARCH_QUERIES = [
    "data scientist", "data analyst", "business analyst",
    "machine learning engineer", "AI engineer",
    "quantitative analyst", "business intelligence analyst",
    "prompt engineer", "data engineer"
]

# Counters to track how many irrelevant jobs we skip
skipped_counts = {'RemoteOK': 0, 'LinkedIn': 0, 'WeWorkRemotely': 0, 'Indeed': 0}

print(f"Relevance filter active: {len(RELEVANT_KEYWORDS)} keywords")
print(f"Search queries: {len(SEARCH_QUERIES)}")

Relevance filter active: 15 keywords
Search queries: 9


---
## 3. Source 1: RemoteOK API (Free JSON API)

RemoteOK provides a free public JSON API — no HTML parsing needed. This is the most reliable source.

**Data types collected:** strings, floats, dates, lists (tags)

In [3]:
def scrape_remoteok(tags_list, delay=2):
    """
    Scrape jobs from RemoteOK's free JSON API.
    Only keeps listings whose titles pass the relevance filter.
    """
    all_jobs = []
    seen_ids = set()
    
    for tag in tags_list:
        try:
            url = f"https://remoteok.com/api?tags={tag}"
            resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=20)
            
            if resp.status_code == 200:
                data = resp.json()
                # First item is metadata, skip it
                jobs = data[1:] if len(data) > 1 else []
                
                for job in jobs:
                    job_id = job.get('id', '')
                    if job_id in seen_ids:
                        continue
                    seen_ids.add(job_id)
                    
                    title = job.get('position', '')
                    
                    # Relevance filter — skip non-matching titles
                    if not is_relevant(title):
                        skipped_counts['RemoteOK'] += 1
                        continue
                    
                    all_jobs.append({
                        'title': title,
                        'company': job.get('company', ''),
                        'location': job.get('location', 'Remote'),
                        'salary_min': job.get('salary_min'),
                        'salary_max': job.get('salary_max'),
                        'tags': ', '.join(job.get('tags', [])),
                        'date_posted': job.get('date', ''),
                        'url': job.get('url', ''),
                        'description': job.get('description', '')[:500],
                        'source': 'RemoteOK',
                        'search_tag': tag
                    })
                
                print(f"  [{tag}] Found {len(jobs)} listings, kept relevant → {len(all_jobs)} total")
            else:
                print(f"  [{tag}] HTTP {resp.status_code}")
                
        except Exception as e:
            print(f"  [{tag}] Error: {e}")
        
        time.sleep(delay)
    
    return pd.DataFrame(all_jobs)

print("RemoteOK scraper ready.")

RemoteOK scraper ready.


In [4]:
# Run RemoteOK scraper
remoteok_tags = [
    'data-science', 'data-analyst', 'machine-learning',
    'ai', 'artificial-intelligence', 'deep-learning',
    'nlp', 'data-engineer', 'business-analyst',
    'python', 'analytics', 'quantitative'
]

print("Scraping RemoteOK...")
df_remoteok = scrape_remoteok(remoteok_tags)
print(f"\nTotal RemoteOK jobs kept: {len(df_remoteok)}")
print(f"Skipped (irrelevant): {skipped_counts['RemoteOK']}")
df_remoteok.head()

Scraping RemoteOK...
  [data-science] Found 0 listings, kept relevant → 0 total
  [data-analyst] Found 0 listings, kept relevant → 0 total
  [machine-learning] Found 0 listings, kept relevant → 0 total
  [ai] Found 98 listings, kept relevant → 26 total
  [artificial-intelligence] Found 0 listings, kept relevant → 26 total
  [deep-learning] Found 0 listings, kept relevant → 26 total
  [nlp] Found 0 listings, kept relevant → 26 total
  [data-engineer] Found 0 listings, kept relevant → 26 total
  [business-analyst] Found 0 listings, kept relevant → 26 total
  [python] Found 70 listings, kept relevant → 60 total
  [analytics] Found 99 listings, kept relevant → 92 total
  [quantitative] Found 0 listings, kept relevant → 92 total

Total RemoteOK jobs kept: 92
Skipped (irrelevant): 159


,title,company,location,salary_min,salary_max,tags,date_posted,url,description,source,search_tag
0,Senior Site Reliability Engineer AI Infrastructure,Andromeda Cluster,San Francisco,0,0,"design, training, technical, software, code, manager, financial, cloud, mana...",2026-04-29T00:01:03+00:00,https://remoteOK.com/remote-jobs/remote-senior-site-reliability-engineer-ai-...,"<p style=""min-height:1.5em""><strong>Senior Site Reliability Engineer - AI In...",RemoteOK,ai
1,HQ AI Enablement Lead,Everfield,,0,0,"saas, teaching, training, consulting, consult, technical, support, software,...",2026-04-25T00:00:49+00:00,https://remoteOK.com/remote-jobs/remote-hq-ai-enablement-lead-everfield-1131322,"<h2><strong>About Everfield</strong></h2><p style=""min-height:1.5em"">Everfie...",RemoteOK,ai
2,Mid Senior AI Video Artist,EverAI,,30000,60000,"marketing, video, ai",2026-04-23T08:11:46+00:00,https://remoteOK.com/remote-jobs/remote-mid-senior-ai-video-artist-everai-11...,<h1>Our Vision &amp; Products</h1><p><em><strong>ð EverAI â Building th...,RemoteOK,ai
3,Search Engine Evaluation Specialist Freelance AI Trainer Project,Invisible Agency,United States of America,0,0,"trainer, design, training, digital nomad",2026-04-22T00:01:28+00:00,https://remoteOK.com/remote-jobs/remote-search-engine-evaluation-specialist-...,<div><h3><strong>Project Overview</strong></h3>\n<p>We are sourcing independ...,RemoteOK,ai
4,Data Analyst,Criptoro,,70000,80000,"other, analyst, crypto, defi, blockchain, web3",2026-04-19T22:35:44+00:00,https://remoteOK.com/remote-jobs/remote-data-analyst-criptoro-1131219,<p>We are a Web3-driven company building decentralized products and working ...,RemoteOK,ai


---
## 4. Source 2: LinkedIn Public Job Search

LinkedIn's public job search pages (no login required) can be scraped with care. We add delays between requests to avoid rate limiting, as noted in Prof. Wang's notebook.

**Note:** LinkedIn may block requests after too many. If this happens, wait 5–10 minutes and retry with fewer queries.

In [5]:
def scrape_linkedin(search_queries, locations=None, max_pages=3, delay=5):
    """
    Scrape LinkedIn's public job search (no login required).
    Only keeps listings whose titles pass the relevance filter.
    """
    if locations is None:
        locations = [
            "San Francisco Bay Area", "New York", "Los Angeles",
            "Seattle", "Austin", "Chicago", "Boston", "Remote"
        ]
    
    all_jobs = []
    seen_urls = set()
    
    for query in search_queries:
        for location in locations:
            for page in range(max_pages):
                start = page * 25  # LinkedIn paginates by 25
                try:
                    url = (
                        f"https://www.linkedin.com/jobs/search/"
                        f"?keywords={quote_plus(query)}"
                        f"&location={quote_plus(location)}"
                        f"&start={start}"
                    )
                    
                    resp = requests.get(url, headers=HEADERS, timeout=20)
                    
                    if resp.status_code != 200:
                        print(f"  [{query} | {location}] HTTP {resp.status_code} - skipping")
                        break
                    
                    soup = BeautifulSoup(resp.text, 'html.parser')
                    cards = soup.find_all('div', class_='base-card')
                    
                    if not cards:
                        break  # no more results
                    
                    for card in cards:
                        title_el = card.find('h3', class_='base-search-card__title')
                        company_el = card.find('h4', class_='base-search-card__subtitle')
                        loc_el = card.find('span', class_='job-search-card__location')
                        link_el = card.find('a', class_='base-card__full-link')
                        time_el = card.find('time')
                        
                        title_text = title_el.text.strip() if title_el else ''
                        
                        # Relevance filter — skip non-matching titles
                        if not is_relevant(title_text):
                            skipped_counts['LinkedIn'] += 1
                            continue
                        
                        job_url = link_el['href'].split('?')[0] if link_el else ''
                        if job_url in seen_urls:
                            continue
                        seen_urls.add(job_url)
                        
                        all_jobs.append({
                            'title': title_text,
                            'company': company_el.text.strip() if company_el else '',
                            'location': loc_el.text.strip() if loc_el else '',
                            'date_posted': time_el.get('datetime', '') if time_el else '',
                            'url': job_url,
                            'source': 'LinkedIn',
                            'search_query': query,
                            'search_location': location
                        })
                    
                except Exception as e:
                    print(f"  [{query} | {location}] Error: {e}")
                    break
                
                # Random delay to avoid rate limiting
                time.sleep(delay + random.uniform(1, 3))
            
            print(f"  [{query} | {location}] → {len(all_jobs)} total relevant jobs")
    
    return pd.DataFrame(all_jobs)

print("LinkedIn scraper ready.")

LinkedIn scraper ready.


In [6]:
# Run LinkedIn scraper
linkedin_queries = [
    "data scientist", "data analyst", "machine learning engineer",
    "AI engineer", "business analyst", "quantitative analyst",
    "business intelligence analyst", "prompt engineer"
]

linkedin_locations = [
    "San Francisco Bay Area", "New York", "Los Angeles",
    "Seattle", "Austin", "Chicago", "Remote"
]

print("Scraping LinkedIn (this may take 10-15 minutes)...")
print("If you get blocked, reduce queries or wait and retry.\n")
df_linkedin = scrape_linkedin(linkedin_queries, linkedin_locations, max_pages=2, delay=4)
print(f"\nTotal LinkedIn jobs kept: {len(df_linkedin)}")
print(f"Skipped (irrelevant): {skipped_counts['LinkedIn']}")
df_linkedin.head()

Scraping LinkedIn (this may take 10-15 minutes)...
If you get blocked, reduce queries or wait and retry.

  [data scientist | San Francisco Bay Area] → 53 total relevant jobs
  [data scientist | New York] → 109 total relevant jobs
  [data scientist | Los Angeles] → 186 total relevant jobs
  [data scientist | Seattle] → 230 total relevant jobs
  [data scientist | Austin] → 245 total relevant jobs
  [data scientist | Chicago] → 302 total relevant jobs
  [data scientist | Remote] → 355 total relevant jobs
  [data analyst | San Francisco Bay Area] → 410 total relevant jobs
  [data analyst | New York] → 464 total relevant jobs
  [data analyst | Los Angeles] → 518 total relevant jobs
  [data analyst | Seattle] → 575 total relevant jobs
  [data analyst | Austin] → 577 total relevant jobs
  [data analyst | Chicago] → 630 total relevant jobs
  [data analyst | Remote] → 681 total relevant jobs
  [machine learning engineer | San Francisco Bay Area] → 715 total relevant jobs
  [machine learning en

,title,company,location,date_posted,url,source,search_query,search_location
0,Data Scientist,Samba TV,"San Francisco, CA",2026-04-14,https://www.linkedin.com/jobs/view/data-scientist-at-samba-tv-4401794091,LinkedIn,data scientist,San Francisco Bay Area
1,Machine Learning Engineer Intern,Tinder,"Palo Alto, CA",2026-04-10,https://www.linkedin.com/jobs/view/machine-learning-engineer-intern-at-tinde...,LinkedIn,data scientist,San Francisco Bay Area
2,"Data Scientist, Analytics",Discord,San Francisco Bay Area,2026-04-29,https://www.linkedin.com/jobs/view/data-scientist-analytics-at-discord-43856...,LinkedIn,data scientist,San Francisco Bay Area
3,Data Science - Payments or Fraud Analytics,Straive,"San Jose, CA",2026-04-28,https://www.linkedin.com/jobs/view/data-science-payments-or-fraud-analytics-...,LinkedIn,data scientist,San Francisco Bay Area
4,"Data Scientist, Product Analytics",Meta,"Menlo Park, CA",2026-04-15,https://www.linkedin.com/jobs/view/data-scientist-product-analytics-at-meta-...,LinkedIn,data scientist,San Francisco Bay Area


---
## 5. Source 3: WeWorkRemotely

Server-side rendered HTML — straightforward to scrape (per Prof. Wang's notebook). Good for remote-focused roles.

In [7]:
def scrape_weworkremotely():
    """
    Scrape job listings from WeWorkRemotely.
    Only keeps listings whose titles pass the relevance filter.
    """
    categories = [
        "https://weworkremotely.com/categories/remote-back-end-programming-jobs",
        "https://weworkremotely.com/categories/remote-full-stack-programming-jobs",
        "https://weworkremotely.com/categories/remote-data-jobs",
        "https://weworkremotely.com/remote-jobs/search?term=data+scientist",
        "https://weworkremotely.com/remote-jobs/search?term=machine+learning",
        "https://weworkremotely.com/remote-jobs/search?term=data+analyst",
        "https://weworkremotely.com/remote-jobs/search?term=AI",
    ]
    
    all_jobs = []
    seen_links = set()
    
    for cat_url in categories:
        try:
            resp = requests.get(cat_url, headers=HEADERS, timeout=20)
            soup = BeautifulSoup(resp.text, 'html.parser')
            
            for li in soup.find_all('li', class_='feature'):
                link_el = li.find('a', href=True)
                if not link_el:
                    continue
                
                href = link_el.get('href', '')
                if '/remote-jobs/' not in href:
                    continue
                
                full_url = urljoin('https://weworkremotely.com', href)
                if full_url in seen_links:
                    continue
                seen_links.add(full_url)
                
                company_el = li.find('span', class_='company')
                title_el = li.find('span', class_='title')
                region_el = li.find('span', class_='region')
                
                title_text = title_el.text.strip() if title_el else ''
                
                # Relevance filter — skip non-matching titles
                if not is_relevant(title_text):
                    skipped_counts['WeWorkRemotely'] += 1
                    continue
                
                all_jobs.append({
                    'title': title_text,
                    'company': company_el.text.strip() if company_el else '',
                    'location': region_el.text.strip() if region_el else 'Remote',
                    'url': full_url,
                    'source': 'WeWorkRemotely',
                    'search_category': cat_url.split('/')[-1]
                })
            
            print(f"  [{cat_url.split('/')[-1][:40]}] → {len(all_jobs)} total relevant")
            
        except Exception as e:
            print(f"  Error: {e}")
        
        time.sleep(2)
    
    return pd.DataFrame(all_jobs)

print("WeWorkRemotely scraper ready.")

WeWorkRemotely scraper ready.


In [8]:
# Run WeWorkRemotely scraper
print("Scraping WeWorkRemotely...")
df_wwr = scrape_weworkremotely()
print(f"\nTotal WeWorkRemotely jobs kept: {len(df_wwr)}")
print(f"Skipped (irrelevant): {skipped_counts['WeWorkRemotely']}")
if len(df_wwr) > 0:
    display(df_wwr.head())

Scraping WeWorkRemotely...
  [remote-back-end-programming-jobs] → 0 total relevant
  [remote-full-stack-programming-jobs] → 0 total relevant
  [remote-data-jobs] → 0 total relevant
  [search?term=data+scientist] → 0 total relevant
  [search?term=machine+learning] → 0 total relevant
  [search?term=data+analyst] → 0 total relevant
  [search?term=AI] → 0 total relevant

Total WeWorkRemotely jobs kept: 0
Skipped (irrelevant): 2


---
## 6. Source 4: Indeed (Backup Source)

Use this if you need more listings to reach 500+. Indeed is more aggressive with blocking, so use generous delays.

**⚠️ Run this only if your total from Sources 1-3 is below 400.**

In [9]:
def scrape_indeed(search_queries, locations=None, max_pages=2, delay=6):
    """
    Scrape job listings from Indeed search results.
    Only keeps listings whose titles pass the relevance filter.
    """
    if locations is None:
        locations = ["San Francisco, CA", "New York, NY", "Remote"]
    
    all_jobs = []
    seen_titles = set()
    
    for query in search_queries:
        for location in locations:
            for page in range(max_pages):
                start = page * 10
                try:
                    url = (
                        f"https://www.indeed.com/jobs"
                        f"?q={quote_plus(query)}"
                        f"&l={quote_plus(location)}"
                        f"&start={start}"
                    )
                    
                    resp = requests.get(url, headers=HEADERS, timeout=20)
                    if resp.status_code != 200:
                        print(f"  [{query} | {location}] HTTP {resp.status_code}")
                        break
                    
                    soup = BeautifulSoup(resp.text, 'html.parser')
                    
                    job_cards = soup.find_all('div', class_='job_seen_beacon')
                    if not job_cards:
                        job_cards = soup.find_all('div', attrs={'data-jk': True})
                    
                    for card in job_cards:
                        title_el = card.find('h2') or card.find('a', attrs={'data-jk': True})
                        company_el = card.find('span', attrs={'data-testid': 'company-name'})
                        if not company_el:
                            company_el = card.find('span', class_='companyName')
                        loc_el = card.find('div', attrs={'data-testid': 'text-location'})
                        if not loc_el:
                            loc_el = card.find('div', class_='companyLocation')
                        salary_el = card.find('div', class_='salary-snippet-container')
                        if not salary_el:
                            salary_el = card.find('div', attrs={'data-testid': 'attribute_snippet_testid'})
                        
                        title_text = title_el.text.strip() if title_el else ''
                        
                        # Relevance filter — skip non-matching titles
                        if not is_relevant(title_text):
                            skipped_counts['Indeed'] += 1
                            continue
                        
                        dedup_key = f"{title_text}|{company_el.text.strip() if company_el else ''}"
                        if dedup_key in seen_titles:
                            continue
                        seen_titles.add(dedup_key)
                        
                        all_jobs.append({
                            'title': title_text,
                            'company': company_el.text.strip() if company_el else '',
                            'location': loc_el.text.strip() if loc_el else '',
                            'salary_text': salary_el.text.strip() if salary_el else '',
                            'source': 'Indeed',
                            'search_query': query,
                            'search_location': location
                        })
                    
                except Exception as e:
                    print(f"  [{query} | {location}] Error: {e}")
                    break
                
                time.sleep(delay + random.uniform(2, 5))
        
            print(f"  [{query} | {location}] → {len(all_jobs)} total relevant")
    
    return pd.DataFrame(all_jobs)

print("Indeed scraper ready (use as backup if needed).")

Indeed scraper ready (use as backup if needed).


In [10]:
# Uncomment and run ONLY if you need more data to reach 500+

# print("Scraping Indeed (backup source)...")
# indeed_queries = ["data scientist", "data analyst", "machine learning engineer", "AI engineer"]
# indeed_locations = ["San Francisco, CA", "New York, NY", "Austin, TX", "Seattle, WA", "Remote"]
# df_indeed = scrape_indeed(indeed_queries, indeed_locations, max_pages=3, delay=5)
# print(f"\nTotal Indeed jobs kept: {len(df_indeed)}")
# print(f"Skipped (irrelevant): {skipped_counts['Indeed']}")
# df_indeed.head()

---
## 7. Combine All Sources

In [11]:
# Standardize columns across all sources before merging
standard_cols = [
    'title', 'company', 'location', 'salary_min', 'salary_max',
    'salary_text', 'tags', 'date_posted', 'url', 'source', 'description'
]

def standardize_df(df, source_name):
    """Add missing standard columns with NaN."""
    for col in standard_cols:
        if col not in df.columns:
            df[col] = np.nan
    df = df[standard_cols].copy()
    return df

# Combine all DataFrames
frames = []

if len(df_remoteok) > 0:
    frames.append(standardize_df(df_remoteok, 'RemoteOK'))
    
if len(df_linkedin) > 0:
    frames.append(standardize_df(df_linkedin, 'LinkedIn'))
    
if len(df_wwr) > 0:
    frames.append(standardize_df(df_wwr, 'WeWorkRemotely'))

# Uncomment if Indeed was used
# if len(df_indeed) > 0:
#     frames.append(standardize_df(df_indeed, 'Indeed'))

df_raw = pd.concat(frames, ignore_index=True)

print(f"Total relevant listings collected: {len(df_raw)}")
print(f"\nListings by source:")
print(df_raw['source'].value_counts())
print(f"\nIrrelevant jobs filtered out during scraping:")
for src, count in skipped_counts.items():
    if count > 0:
        print(f"  - {src}: {count} skipped")
print(f"\n{'✅' if len(df_raw) >= 500 else '⚠️'} {'Met' if len(df_raw) >= 500 else 'Below'} 500 listing minimum")

Total relevant listings collected: 2011

Listings by source:
source
LinkedIn    1919
RemoteOK      92
Name: count, dtype: int64

Irrelevant jobs filtered out during scraping:
  - RemoteOK: 159 skipped
  - LinkedIn: 544 skipped
  - WeWorkRemotely: 2 skipped

✅ Met 500 listing minimum


In [12]:
# Save raw data as checkpoint
df_raw.to_csv('job_listings_raw.csv', index=False)
print(f"Saved {len(df_raw)} raw listings to job_listings_raw.csv")
df_raw.info()

Saved 2011 raw listings to job_listings_raw.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2011 entries, 0 to 2010
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        2011 non-null   object 
 1   company      2011 non-null   object 
 2   location     2011 non-null   object 
 3   salary_min   92 non-null     float64
 4   salary_max   92 non-null     float64
 5   salary_text  0 non-null      float64
 6   tags         92 non-null     object 
 7   date_posted  2011 non-null   object 
 8   url          2011 non-null   object 
 9   source       2011 non-null   object 
 10  description  92 non-null     object 
dtypes: float64(3), object(8)
memory usage: 172.9+ KB


---
## 8. Data Cleaning & Preparation

Per project instructions: handle missing data, outliers, incorrect records, redundant records, and data type constraints.

In [13]:
# Load from checkpoint (can restart from here)
df = pd.read_csv('job_listings_raw.csv')
print(f"Loaded {len(df)} raw records")
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nDuplicates (by title + company): {df.duplicated(subset=['title', 'company']).sum()}")

Loaded 2011 raw records

Missing values:
title             0
company           0
location         30
salary_min     1919
salary_max     1919
salary_text    2011
tags           1919
date_posted       0
url               0
source            0
description    1919
dtype: int64

Duplicates (by title + company): 257


### 8.1 Remove Duplicates & Empty Rows

In [14]:
before = len(df)

# Remove exact duplicates
df = df.drop_duplicates(subset=['title', 'company'], keep='first')

# Remove listings with no title
df = df[df['title'].str.strip().ne('') & df['title'].notna()].copy()

print(f"Removed {before - len(df)} duplicate/empty records")
print(f"Remaining: {len(df)}")

Removed 257 duplicate/empty records
Remaining: 1754


### 8.2 Standardize Job Titles into Categories

In [15]:
def classify_job_title(title):
    """Classify a job title into a standard category."""
    title_lower = title.lower()
    
    # AI-related (check first — more specific)
    if any(kw in title_lower for kw in ['machine learning', 'ml engineer', 'ml ']):
        return 'Machine Learning Engineer'
    if any(kw in title_lower for kw in ['ai engineer', 'ai research', 'artificial intelligence']):
        return 'AI Engineer / Researcher'
    if 'prompt engineer' in title_lower:
        return 'Prompt Engineer'
    if any(kw in title_lower for kw in ['deep learning', 'nlp', 'computer vision', 'cv engineer']):
        return 'ML/AI Specialist'
    
    # Data Science
    if 'data scientist' in title_lower:
        return 'Data Scientist'
    if 'data analyst' in title_lower:
        return 'Data Analyst'
    if any(kw in title_lower for kw in ['data engineer', 'data infrastructure']):
        return 'Data Engineer'
    if any(kw in title_lower for kw in ['quantitative', 'quant ']):
        return 'Quantitative Analyst'
    if any(kw in title_lower for kw in ['business intelligence', 'bi analyst', 'bi developer']):
        return 'Business Intelligence Analyst'
    if 'business analyst' in title_lower:
        return 'Business Analyst'
    if 'analytics' in title_lower:
        return 'Analytics / Other'
    
    # Catch-all for anything that passed the relevance filter
    # but doesn't match a specific category
    if 'data' in title_lower:
        return 'Data (General)'
    if 'ai' in title_lower:
        return 'AI (General)'
    
    return 'Other'

def classify_domain(category):
    """Classify into the two project domains."""
    ai_cats = ['Machine Learning Engineer', 'AI Engineer / Researcher', 
               'Prompt Engineer', 'ML/AI Specialist', 'AI (General)']
    ds_cats = ['Data Scientist', 'Data Analyst', 'Data Engineer',
               'Quantitative Analyst', 'Business Intelligence Analyst',
               'Business Analyst', 'Analytics / Other', 'Data (General)']
    if category in ai_cats:
        return 'AI'
    elif category in ds_cats:
        return 'Data Science'
    return 'Other'

df['job_category'] = df['title'].apply(classify_job_title)
df['domain'] = df['job_category'].apply(classify_domain)

# Check if any still fell through to "Other"
other_count = (df['domain'] == 'Other').sum()
if other_count > 0:
    print(f"⚠️ {other_count} jobs still classified as 'Other' — dropping these:")
    print(df[df['domain'] == 'Other']['title'].head(10).tolist())
    df = df[df['domain'] != 'Other'].copy()
else:
    print("✅ All jobs classified into DS or AI domains")

print(f"\nJob category distribution:")
print(df['job_category'].value_counts())
print(f"\nDomain distribution:")
print(df['domain'].value_counts())

⚠️ 156 jobs still classified as 'Other' — dropping these:
['Cyber Security Analyst', 'Senior FinCrime Support Analyst', 'Payments &amp; Billing Operations Analyst', 'Compensation Analyst', 'Financial Crimes Analyst I', 'Sales Enablement Analyst Broker API', 'Senior Open Source Intelligence Analyst', 'Senior Actuary Analyst', 'Social Media Analyst Platform', 'Sales Operations Analyst']

Job category distribution:
job_category
Data Analyst                     285
Machine Learning Engineer        252
Business Analyst                 252
Data Scientist                   218
Quantitative Analyst             161
AI (General)                     105
Business Intelligence Analyst     95
AI Engineer / Researcher          73
Data (General)                    50
Prompt Engineer                   44
Data Engineer                     32
Analytics / Other                 31
Name: count, dtype: int64

Domain distribution:
domain
Data Science    1124
AI               474
Name: count, dtype: int64


### 8.3 Clean Location Data

In [16]:
def classify_work_type(row):
    """Determine remote/hybrid/onsite from location, title, and search context."""
    title = str(row.get('title', '')).lower()
    location = str(row.get('location', '')).lower()
    
    # Check both title and location for remote signals
    remote_keywords = ['remote', 'anywhere', 'worldwide', 'work from home', 'wfh']
    if any(kw in title for kw in remote_keywords) or any(kw in location for kw in remote_keywords):
        return 'Remote'
    if 'hybrid' in title or 'hybrid' in location:
        return 'Hybrid'
    return 'Onsite'

def extract_state(location):
    """Extract US state abbreviation from location string."""
    if pd.isna(location):
        return None
    match = re.search(r',\s*([A-Z]{2})(?:\s|$|,)', str(location))
    if match:
        return match.group(1)
    return None

# Apply row-wise so it checks both title and location
df['work_type'] = df.apply(classify_work_type, axis=1)
df['state'] = df['location'].apply(extract_state)

print("Work type distribution:")
print(df['work_type'].value_counts())
print(f"\nTop states:")
print(df['state'].value_counts().head(10))

Work type distribution:
work_type
Onsite    1552
Remote      37
Hybrid       9
Name: count, dtype: int64

Top states:
state
CA    466
IL    255
NY    186
WA    183
CO      9
TX      7
PA      5
VA      4
OH      2
GA      2
Name: count, dtype: int64


### 8.4 Parse & Clean Salary Data

**Note:** Salary data is sparse — primarily available from RemoteOK. LinkedIn public search pages rarely expose salary. We retain what's available for descriptive reporting but do not build core research questions around salary.

In [17]:
def parse_salary_text(text):
    """Extract min and max salary from a salary text string."""
    if pd.isna(text) or str(text).strip() == '':
        return None, None
    
    text = str(text).replace(',', '').replace('$', '')
    
    # Match ranges like "120000 - 180000" or "120K - 180K"
    range_match = re.findall(r'(\d+\.?\d*)\s*[kK]?', text)
    
    if len(range_match) >= 2:
        vals = [float(x) for x in range_match[:2]]
        if 'k' in text.lower():
            vals = [v * 1000 if v < 1000 else v for v in vals]
        if all(v < 500 for v in vals):
            vals = [v * 2080 for v in vals]  # hourly to annual
        return min(vals), max(vals)
    elif len(range_match) == 1:
        val = float(range_match[0])
        if 'k' in text.lower():
            val *= 1000
        if val < 500:
            val *= 2080
        return val, val
    
    return None, None

# Parse salary_text where salary_min/max aren't already set
mask = df['salary_min'].isna() & df['salary_text'].notna()
parsed = df.loc[mask, 'salary_text'].apply(
    lambda x: pd.Series(parse_salary_text(x), index=['sal_min_parsed', 'sal_max_parsed'])
)
if len(parsed) > 0:
    df.loc[mask, 'salary_min'] = parsed['sal_min_parsed'].values
    df.loc[mask, 'salary_max'] = parsed['sal_max_parsed'].values

# Convert to numeric
df['salary_min'] = pd.to_numeric(df['salary_min'], errors='coerce')
df['salary_max'] = pd.to_numeric(df['salary_max'], errors='coerce')

# Calculate midpoint salary
df['salary_mid'] = df[['salary_min', 'salary_max']].mean(axis=1)

# Remove salary outliers (likely errors)
salary_mask = df['salary_mid'].notna()
df.loc[salary_mask & (df['salary_mid'] < 20000), ['salary_min', 'salary_max', 'salary_mid']] = np.nan
df.loc[salary_mask & (df['salary_mid'] > 500000), ['salary_min', 'salary_max', 'salary_mid']] = np.nan

print(f"Jobs with salary data: {df['salary_mid'].notna().sum()} / {len(df)} ({df['salary_mid'].notna().mean():.1%})")
if df['salary_mid'].notna().sum() > 0:
    print(f"\nSalary summary (annual):")
    print(df['salary_mid'].describe().apply(lambda x: f"${x:,.0f}" if pd.notna(x) else x))

Jobs with salary data: 8 / 1598 (0.5%)

Salary summary (annual):
count          $8
mean     $119,375
std       $65,544
min       $45,000
25%       $73,750
50%       $90,000
75%      $177,500
max      $215,000
Name: salary_mid, dtype: object


### 8.5 Clean Dates & Add Derived Features

In [18]:
# Parse dates
df['date_posted'] = pd.to_datetime(df['date_posted'], errors='coerce')

# Derived features for analysis
df['title_length'] = df['title'].str.len()  # int
df['has_salary'] = df['salary_mid'].notna()  # boolean
df['is_senior'] = df['title'].str.lower().str.contains(
    'senior|sr\\.|lead|principal|staff', na=False
)  # boolean
df['is_entry'] = df['title'].str.lower().str.contains(
    'junior|jr\\.|entry|associate|intern', na=False
)  # boolean

def get_experience_level(row):
    if row['is_senior']:
        return 'Senior'
    if row['is_entry']:
        return 'Entry'
    return 'Mid'

df['experience_level'] = df.apply(get_experience_level, axis=1)

# Description length (proxy for job detail complexity)
df['description_length'] = df['description'].str.len().fillna(0).astype(int)  # int

print("Experience level distribution:")
print(df['experience_level'].value_counts())
print(f"\nData types in final dataset:")
print(df.dtypes)

Experience level distribution:
experience_level
Mid       1244
Senior     250
Entry      104
Name: count, dtype: int64

Data types in final dataset:
title                              object
company                            object
location                           object
salary_min                        float64
salary_max                        float64
salary_text                       float64
tags                               object
date_posted           datetime64[ns, UTC]
url                                object
source                             object
description                        object
job_category                       object
domain                             object
work_type                          object
state                              object
salary_mid                        float64
title_length                        int64
has_salary                           bool
is_senior                            bool
is_entry                             bool
experience_

### 8.6 Final Data Quality Report

In [19]:
print("=" * 60)
print("FINAL DATA QUALITY REPORT")
print("=" * 60)
print(f"Total cleaned records: {len(df)}")
print(f"Minimum met (500+): {'✅ YES' if len(df) >= 500 else '⚠️ NO — run more scrapers'}")
print(f"\nSources: {df['source'].nunique()}")
for src, count in df['source'].value_counts().items():
    print(f"  - {src}: {count}")
print(f"\nDomains:")
for dom, count in df['domain'].value_counts().items():
    print(f"  - {dom}: {count}")
print(f"\nIrrelevant jobs filtered during scraping:")
total_skipped = sum(skipped_counts.values())
print(f"  Total skipped: {total_skipped}")
for src, count in skipped_counts.items():
    if count > 0:
        print(f"  - {src}: {count}")
print(f"\nData types present:")
type_map = {
    'Strings': ['title', 'company', 'location', 'source', 'job_category'],
    'Floats': ['salary_min', 'salary_max', 'salary_mid'],
    'Integers': ['title_length', 'description_length'],
    'Booleans': ['has_salary', 'is_senior', 'is_entry'],
    'Dates': ['date_posted'],
    'Categorical': ['domain', 'work_type', 'experience_level', 'state']
}
for dtype, cols in type_map.items():
    present_cols = [c for c in cols if c in df.columns]
    print(f"  - {dtype}: {', '.join(present_cols)}")
print(f"\nMissing values:")
missing = df.isnull().sum()
missing = missing[missing > 0]
for col, count in missing.items():
    print(f"  - {col}: {count} ({count/len(df):.1%})")

FINAL DATA QUALITY REPORT
Total cleaned records: 1598
Minimum met (500+): ✅ YES

Sources: 2
  - LinkedIn: 1518
  - RemoteOK: 80

Domains:
  - Data Science: 1124
  - AI: 474

Irrelevant jobs filtered during scraping:
  Total skipped: 705
  - RemoteOK: 159
  - LinkedIn: 544
  - WeWorkRemotely: 2

Data types present:
  - Strings: title, company, location, source, job_category
  - Floats: salary_min, salary_max, salary_mid
  - Integers: title_length, description_length
  - Booleans: has_salary, is_senior, is_entry
  - Dates: date_posted
  - Categorical: domain, work_type, experience_level, state

Missing values:
  - location: 25 (1.6%)
  - salary_min: 1590 (99.5%)
  - salary_max: 1590 (99.5%)
  - salary_text: 1598 (100.0%)
  - tags: 1518 (95.0%)
  - date_posted: 1518 (95.0%)
  - description: 1518 (95.0%)
  - state: 469 (29.3%)
  - salary_mid: 1590 (99.5%)


---
## 9. Export Cleaned Dataset

In [21]:
# Select final columns for export
export_cols = [
    'title', 'job_category', 'domain', 'company', 'location', 'state',
    'work_type', 'experience_level', 'salary_min', 'salary_max', 'salary_mid',
    'has_salary', 'is_senior', 'is_entry', 'tags', 'date_posted',
    'title_length', 'description_length', 'source', 'url'
]

df_export = df[[c for c in export_cols if c in df.columns]].copy()

# Strip timezone info for Excel compatibility
if 'date_posted' in df_export.columns:
    df_export['date_posted'] = pd.to_datetime(df_export['date_posted'], errors='coerce')
    if df_export['date_posted'].dt.tz is not None:
        df_export['date_posted'] = df_export['date_posted'].dt.tz_localize(None)

# Save as CSV and Excel
df_export.to_csv('job_listings_cleaned.csv', index=False)
df_export.to_excel('job_listings_cleaned.xlsx', index=False)

print(f"Exported {len(df_export)} cleaned records")
print(f"  → job_listings_cleaned.csv")
print(f"  → job_listings_cleaned.xlsx")
print(f"\nColumns ({len(df_export.columns)}): {', '.join(df_export.columns)}")
print(f"\nPreview:")
df_export.head()

Exported 1598 cleaned records
  → job_listings_cleaned.csv
  → job_listings_cleaned.xlsx

Columns (20): title, job_category, domain, company, location, state, work_type, experience_level, salary_min, salary_max, salary_mid, has_salary, is_senior, is_entry, tags, date_posted, title_length, description_length, source, url

Preview:


,title,job_category,domain,company,location,state,work_type,experience_level,salary_min,salary_max,salary_mid,has_salary,is_senior,is_entry,tags,date_posted,title_length,description_length,source,url
0,Senior Site Reliability Engineer AI Infrastructure,AI (General),AI,Andromeda Cluster,San Francisco,None,Onsite,Senior,NaN,NaN,NaN,False,True,False,"design, training, technical, software, code, manager, financial, cloud, mana...",2026-04-29 00:01:03,50,500,RemoteOK,https://remoteOK.com/remote-jobs/remote-senior-site-reliability-engineer-ai-...
1,HQ AI Enablement Lead,AI (General),AI,Everfield,NaN,None,Onsite,Senior,NaN,NaN,NaN,False,True,False,"saas, teaching, training, consulting, consult, technical, support, software,...",2026-04-25 00:00:49,21,500,RemoteOK,https://remoteOK.com/remote-jobs/remote-hq-ai-enablement-lead-everfield-1131322
2,Mid Senior AI Video Artist,AI (General),AI,EverAI,NaN,None,Onsite,Senior,30000.0,60000.0,45000.0,True,True,False,"marketing, video, ai",2026-04-23 08:11:46,26,500,RemoteOK,https://remoteOK.com/remote-jobs/remote-mid-senior-ai-video-artist-everai-11...
3,Search Engine Evaluation Specialist Freelance AI Trainer Project,AI (General),AI,Invisible Agency,United States of America,None,Onsite,Mid,NaN,NaN,NaN,False,False,False,"trainer, design, training, digital nomad",2026-04-22 00:01:28,64,500,RemoteOK,https://remoteOK.com/remote-jobs/remote-search-engine-evaluation-specialist-...
4,Data Analyst,Data Analyst,Data Science,Criptoro,NaN,None,Onsite,Mid,70000.0,80000.0,75000.0,True,False,False,"other, analyst, crypto, defi, blockchain, web3",2026-04-19 22:35:44,12,500,RemoteOK,https://remoteOK.com/remote-jobs/remote-data-analyst-criptoro-1131219


---
## 10. Research Questions for Notebook 2

Based on the data collected, here are research questions the team will explore in the analysis notebook. These focus on **demand-side analysis** where our data coverage is strongest.

1. **DS vs AI Demand by City:** What is the ratio of Data Science to AI roles across major metros? Are certain cities more AI-heavy?

2. **Experience Level Gap:** What percentage of AI roles are entry-level vs. senior, compared to DS roles? (Barrier to entry in AI)

3. **Geographic Concentration:** Which metros dominate hiring for each role category? Is AI hiring more geographically concentrated than DS hiring?

4. **Role Evolution:** How do emerging roles (Prompt Engineer, ML/AI Specialist) compare in volume and distribution to established ones (Data Analyst, Business Analyst)?

5. **Job Title Complexity:** How does title length and seniority signaling differ across role categories and domains? (e.g., do AI roles use more inflated titles?)


**Note on salary:** Salary data was available for only a small fraction of listings (primarily from RemoteOK). We include descriptive salary statistics where available but do not build core research questions around salary comparisons.

---

### Team Task Division Suggestion

| Member | Responsibility |
|--------|---------------|
| Member 1 | Research Q1 (DS vs AI demand by city) + Q2 (Remote work) |
| Member 2 | Research Q3 (Experience levels) + Q4 (Geographic concentration) |
| Member 3 | Research Q5 (Role evolution) + descriptive salary statistics |
| Member 4 | Presentation slides + overall narrative + summary statistics |